In [10]:
import requests
import matplotlib.pyplot as plt
import numpy as np

In [11]:
# url = "http://127.0.0.1:8000/",
url = "https://gd-secret-api.onrender.com/"

evaluate_url = url + "evaluate"
evaluate_with_grad_url = url + "evaluate-with-gradient"
discovery_url = url + "problems"


In [12]:

def _is_json_like(obj):
    return isinstance(obj, (dict, list))


def test_evaluate_default_x_returns_json():
    resp = requests.post(
        evaluate_url,
        json={"problem_id": "parabola_1d", "x": [2.5]},
    )
    assert resp.status_code == 200
    data = resp.json()
    assert _is_json_like(data)


In [13]:
import requests
import pandas as pd

# API endpoints
base_url = "https://gd-secret-api.onrender.com"

evaluate_url = f"{base_url}/evaluate"
gradient_url = f"{base_url}/evaluate-with-gradient"
problems_url = f"{base_url}/problems"


# 1. Ask the server: "Which problems do you have?"
problems = requests.get(problems_url).json()

print("Available problems:")
for problem_id, info in problems.items():
    print(f"- {problem_id}: {info['name']} ({info['dimension']}D)")
    print(f"  {info['description']}")


Available problems:
- parabola_1d: Simple 1D parabola (1D)
  Easy convex function. Good first gradient descent example.
- wavy_1d: Wavy 1D function (1D)
  Has local wiggles. Good for showing local minima and learning-rate issues.
- bowl_2d: Simple 2D bowl (2D)
  Easy 2D convex function. Good for visualizing movement on a surface.


In [14]:
# 2. Evaluate one point on a simple 1D function
response = requests.post(
    evaluate_url,
    json={
        "problem_id": "parabola_1d",
        "x": [2.5]
    }
)

data = response.json()

print("\nSingle API call:")
print(data)


KeyboardInterrupt: 

In [ ]:
# 3. Try several x values and collect the answers
rows = []

for x_value in [-1, 0, 1, 2, 2.5, 3, 4, 5]:
    response = requests.post(
        evaluate_url,
        json={
            "problem_id": "parabola_1d",
            "x": [x_value]
        }
    )

    data = response.json()

    rows.append({
        "x": x_value,
        "y = f(x)": data["y"]
    })

df = pd.DataFrame(rows)

print("\nEvaluating the function at several points:")
display(df)


# 4. Ask the API for the gradient too
response = requests.post(
    gradient_url,
    json={
        "problem_id": "parabola_1d",
        "x": [2.5]
    }
)

data = response.json()

print("\nFunction value + gradient:")
print(data)

print(
    f"\nAt x = {data['x'][0]}, "
    f"the function value is {data['y']}, "
    f"and the gradient is {data['gradient'][0]}."
)


Evaluating the function at several points:


,x,y = f(x)
0,-1.0,16.00
1,0.0,9.00
2,1.0,4.00
3,2.0,1.00
4,2.5,0.25
5,3.0,0.00
6,4.0,1.00
7,5.0,4.00



Function value + gradient:
{'problem_id': 'parabola_1d', 'x': [2.5], 'y': 0.25, 'gradient': [-1.0]}

At x = 2.5, the function value is 0.25, and the gradient is -1.0.


In [ ]:
def test_evaluate_parabola_1d_returns_json():
    resp = requests.post(
        evaluate_url,
        json={
            "problem_id": "parabola_1d",
            "x": [2.5]
        },
    )
    assert resp.status_code == 200
    data = resp.json()
    assert _is_json_like(data)


In [ ]:
def evaluate_1d(x):
    if isinstance(x, (int, float)):
        x = [x]
    if isinstance(x, np.ndarray):
        x = x.tolist()
    resp = requests.post(
        evaluate_url,
        json={
            "problem_id": "parabola_1d",
            "x": x
        },
    )
    assert resp.status_code == 200
    data = resp.json()
    assert _is_json_like(data)
    return data["y"]


def evaluate_1d_calc_grad(x, eps=1e-6):
    if isinstance(x, (int, float)):
        x = np.array([x])
    x_plus_eps = (x + eps).tolist()
    x_minus_eps = (x - eps).tolist()
    # convert x_plus_eps and x_minus_eps to lists for the API
    y_plus = evaluate_1d(x_plus_eps)
    y_minus = evaluate_1d(x_minus_eps)
    y = evaluate_1d(x.tolist())
    grad = (y_plus - y_minus) / (2 * eps)
    return y, grad


def evaluate_2d(x_vec):
    """Evaluate a 2D problem by POSTing to the evaluate endpoint."""
    resp = requests.post(
        evaluate_url,
        json={
            "problem_id": "bowl_2d",
            "x": x_vec,
        },
    )
    assert resp.status_code == 200
    data = resp.json()
    assert _is_json_like(data)
    return data["y"]

def evaluate_calc_grad(x_vec, api_func, eps=1e-6):
    """Compute numerical gradient for a 2D input using central differences."""
    # ensure we have a mutable copy
    grad = np.zeros_like(x_vec)
    for i in range(x_vec.shape[0]):
        x_plus = list(x_vec)
        x_minus = list(x_vec)
        x_plus[i] += eps
        x_minus[i] -= eps
        y_plus = api_func(x_plus)
        y_minus = api_func(x_minus)
        grad[i] = (y_plus - y_minus) / (2 * eps)
    y = api_func(x_vec)
    return y, grad

# print(evaluate_1d(2.5))
# print(evaluate_1d_calc_grad(2.5))
# print(evaluate_2d([10.0, -3.0]))
# print(evaluate_2d_calc_grad([10.0, -3.0]))

In [ ]:
curr_diff = np.inf
curr_x = np.array([np.random.rand() * 10])
step_size = 0.01


print(f"Initial x: {curr_x}")
print(f"Initial y: {evaluate_1d(curr_x)}")
print(f"Step size: {step_size}")

y_values = []
grad_sizes = []

while curr_diff > 1e-5:
    # y, grad = evaluate_1d_calc_grad(curr_x)
    y, grad = evaluate_calc_grad(curr_x, evaluate_1d)
    y_values.append(y)
    grad = np.array(grad)
    grad_sizes.append(np.linalg.norm(grad))
    print(f"x: {curr_x}, y: {y}, grad: {grad}")
    next_x = curr_x - step_size * grad
    curr_diff = abs(next_x - curr_x)
    curr_x = next_x

print(f"Final x: {curr_x}, Final y: {evaluate_1d(curr_x)}")

Initial x: [6.47607378]
Initial y: 12.083088930572544
Step size: 0.01
x: [6.47607378], y: 12.083088930572544, grad: [6.95214756]
x: [6.40655231], y: 11.604598608818266, grad: [6.81310461]
x: [6.33842126], y: 11.14505650387395, grad: [6.67684252]
x: [6.27165283], y: 10.703712266231502, grad: [6.54330567]
x: [6.20621978], y: 10.279845260434508, grad: [6.41243956]
x: [6.14209538], y: 9.872763388075974, grad: [6.28419076]
x: [6.07925347], y: 9.481801957865931, grad: [6.15850695]
x: [6.0176684], y: 9.106322600297341, grad: [6.03533681]
x: [5.95731504], y: 8.745712225289209, grad: [5.91463007]
x: [5.89816874], y: 8.399382021130066, grad: [5.79633747]
x: [5.84020536], y: 8.066766493006934, grad: [5.68041072]
x: [5.78340125], y: 7.747322539827885, grad: [5.56680251]
x: [5.72773323], y: 7.440528567191969, grad: [5.45546646]
x: [5.67317856], y: 7.145883635896403, grad: [5.34635713]
x: [5.61971499], y: 6.862906643878975, grad: [5.23942999]
x: [5.56732069], y: 6.591135540729282, grad: [5.13464139]

KeyboardInterrupt: 

In [ ]:
# plot y_values over time
plt.plot(y_values)
plt.plot(grad_sizes)
plt.xlabel("Iteration")
plt.ylabel("y value") 